# Cross-check selected test UPD

Independently recompute the overall and segment UPD values for each validation-selected test recommendation export using the unchanged evaluator from `upd-evaluation.ipynb`. Compare against `testUPD` in `selected_on_validation.csv` at absolute tolerance `1e-9`. This checks only settings selected on validation; it does not re-evaluate the full tuning grid.

Set `calibrationRunId` to a run containing `selected_on_validation.csv`, selected `top-10-*` exports, and the four evaluator input Parquet tables.

In [ ]:
val calibrationRunId = "2026-11-16T00-23-53.926024Z"
val outputsDir = Seq(new java.io.File("outputs"), new java.io.File("../outputs"))
  .find(_.isDirectory)
  .getOrElse(throw new IllegalArgumentException("Run this notebook from the repository root or notebooks/ directory."))
val calibrationDir = new java.io.File(outputsDir, s"calibration/$calibrationRunId")
require(calibrationDir.isDirectory,
  s"Calibration run folder does not exist: ${calibrationDir.getPath}")
val crosscheckPath = new java.io.File(calibrationDir, "upd-crosscheck.csv").toPath
require(!java.nio.file.Files.exists(crosscheckPath),
  s"Cross-check output already exists and will not be overwritten: $crosscheckPath")
println(s"PASS: configured calibration run ${calibrationDir.getPath}")


## Start the independent Spark session

This notebook runs independently of the calibration notebook and starts its own local Spark session. Verify the pinned Spark, Scala, and Java versions before loading data.


In [ ]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("CS 7265 - Baseline UPD Cross-check")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val environmentCheck = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(environmentCheck == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"PASS: Spark environment check on $environmentCheck synthetic rows")


## Load the persisted evaluator inputs

Load the four Parquet datasets exported by the calibration notebook. `evaluateUPD` validates their schemas, key uniqueness, cohort coverage, labels, and list sizes before scoring; these cached DataFrames are reused across all settings.


In [ ]:
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions.col

val inputsDir = new java.io.File(calibrationDir, "inputs")
def loadInput(name: String): DataFrame =
  spark.read.parquet(new java.io.File(inputsDir, name).getPath).cache()

val profiles = loadInput("user-popularity-profiles")
val movieGroups = loadInput("movie-popularity-groups")
val segments = loadInput("user-segments")
val evaluationUsers = loadInput("evaluation-users")

val inputCounts = Seq(
  "user-popularity-profiles" -> profiles.count(),
  "movie-popularity-groups" -> movieGroups.count(),
  "user-segments" -> segments.count(),
  "evaluation-users" -> evaluationUsers.count())
assert(inputCounts.forall(_._2 > 0L), s"Expected nonempty evaluator inputs: $inputCounts")
val inputSummary = inputCounts.map { case (name, count) => s"$name=$count" }.mkString(", ")
println(s"PASS: loaded evaluator inputs: $inputSummary")


## Validated evaluator helpers

The following validation and scoring definitions are copied unchanged from `upd-evaluation.ipynb`. They preserve the evaluator's base-2 Jensen–Shannon divergence and equal-weight average of the three segment means.


In [ ]:
import spark.implicits._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions.{avg, col, count, isnan, lit}

def validateUserMismatchScores(scores: DataFrame): Unit = {
  val invalidRows = scores.filter(
    col("userId").isNull ||
    col("segment").isNull ||
    col("mismatch").isNull ||
    isnan(col("mismatch")) ||
    col("mismatch") < 0.0 ||
    col("mismatch") > 1.0
  )
  assert(invalidRows.count() == 0L,
    "Every row must have a user, a segment, and a finite mismatch score between 0 and 1.")

  val duplicateUsers = scores.groupBy("userId").count()
    .filter(col("count") > 1L)
  assert(duplicateUsers.count() == 0L,
    "Each user must contribute exactly one mismatch score.")

  val expectedSegments = Set("blockbuster-focused", "diverse", "niche-focused")
  val actualSegments = scores.select("segment").distinct()
    .as[String].collect().toSet
  assert(actualSegments == expectedSegments,
    "Evaluation must contain exactly the three expected user segments.")
}


In [ ]:
import org.apache.spark.sql.Column
import org.apache.spark.sql.types.{DataType, IntegerType, DoubleType, StringType}
import org.apache.spark.sql.functions.{abs, greatest, least, log2, sum, when}

def requireColumns(
  table: DataFrame,
  label: String,
  fields: Seq[(String, DataType)]
): Unit = {
  fields.foreach { case (name, dataType) =>
    require(table.columns.count(_ == name) == 1,
      s"$label must contain exactly one column named $name.")
    require(table.schema(name).dataType == dataType,
      s"$label.$name must have type $dataType; convert it explicitly before evaluation.")
  }
}

def requireUniqueKeys(table: DataFrame, keys: Seq[String], label: String): Unit = {
  require(table.filter(keys.map(k => col(k).isNull).reduce(_ || _)).limit(1).count() == 0L,
    s"$label contains missing keys.")
  require(table.groupBy(keys.map(col): _*).count()
    .filter(col("count") > 1L).limit(1).count() == 0L,
    s"$label contains duplicate keys.")
}

def requireUserCoverage(table: DataFrame, expected: DataFrame, label: String): Unit = {
  val actual = table.select("userId").distinct()
  require(expected.join(actual, Seq("userId"), "left_anti").limit(1).count() == 0L,
    s"$label is missing expected users.")
  require(actual.join(expected, Seq("userId"), "left_anti").limit(1).count() == 0L,
    s"$label contains unexpected users; select the evaluation cohort explicitly.")
}

// Builds Spark expressions on the driver; no notebook closure is sent to workers.
def klContributionColumn(proportion: Column, reference: Column): Column = {
  when(proportion === 0.0, lit(0.0))
    .otherwise(proportion * log2(proportion / reference))
}

def evaluateUPD(
  profiles: DataFrame,
  movieGroups: DataFrame,
  recommendations: DataFrame,
  segments: DataFrame,
  expectedUsers: DataFrame,
  listSize: Int = 10
): (DataFrame, DataFrame, DataFrame) = {
  require(listSize > 0, "Recommendation list size must be positive.")
  val epsilon = 1e-9
  requireColumns(profiles, "profiles", Seq(
    "userId" -> IntegerType, "head" -> DoubleType,
    "mid" -> DoubleType, "tail" -> DoubleType))
  requireColumns(movieGroups, "movieGroups", Seq(
    "movieId" -> IntegerType, "popularityGroup" -> StringType))
  requireColumns(recommendations, "recommendations", Seq(
    "userId" -> IntegerType, "movieId" -> IntegerType))
  requireColumns(segments, "segments", Seq("userId" -> IntegerType, "segment" -> StringType))
  requireColumns(expectedUsers, "expectedUsers", Seq("userId" -> IntegerType))

  val cohort = expectedUsers.select("userId")
  val profileData = profiles.select("userId", "head", "mid", "tail")
  val groupData = movieGroups.select("movieId", "popularityGroup")
  val recommendationData = recommendations.select("userId", "movieId")
  val segmentData = segments.select("userId", "segment")

  requireUniqueKeys(cohort, Seq("userId"), "expectedUsers")
  require(cohort.limit(1).count() == 1L, "The evaluation cohort must not be empty.")
  requireUniqueKeys(profileData, Seq("userId"), "profiles")
  requireUniqueKeys(groupData, Seq("movieId"), "movieGroups")
  requireUniqueKeys(segmentData, Seq("userId"), "segments")
  requireUniqueKeys(recommendationData, Seq("userId", "movieId"), "recommendations")
  requireUserCoverage(profileData, cohort, "profiles")
  requireUserCoverage(segmentData, cohort, "segments")
  requireUserCoverage(recommendationData, cohort, "recommendations")

  val invalidProportion = Seq("head", "mid", "tail").map { name =>
    col(name).isNull || isnan(col(name)) || col(name) < 0.0 || col(name) > 1.0
  }.reduce(_ || _)
  require(profileData.filter(invalidProportion ||
    abs(col("head") + col("mid") + col("tail") - 1.0) > epsilon)
    .limit(1).count() == 0L, "Historical mixes must be finite, nonnegative, and sum to 1.")
  require(groupData.filter(col("popularityGroup").isNull ||
    !col("popularityGroup").isin("H", "M", "T")).limit(1).count() == 0L,
    "Movie popularity labels must be H, M, or T.")
  require(segmentData.filter(col("segment").isNull ||
    !col("segment").isin("blockbuster-focused", "diverse", "niche-focused"))
    .limit(1).count() == 0L, "User segment labels are missing or unknown.")
  require(segmentData.select("segment").distinct().count() == 3L,
    "All three user segments must be represented.")
  require(recommendationData.groupBy("userId").count()
    .filter(col("count") =!= listSize.toLong).limit(1).count() == 0L,
    s"Each user must have exactly $listSize distinct recommended movies.")
  require(recommendationData.select("movieId").distinct()
    .join(groupData.select("movieId"), Seq("movieId"), "left_anti")
    .limit(1).count() == 0L, "Some recommended movies have no popularity label.")

  val recommendationMixes = recommendationData.join(groupData, Seq("movieId"), "inner")
    .groupBy("userId")
    .agg(
      count("*").as("recommendationCount"),
      (sum(when(col("popularityGroup") === "H", 1.0).otherwise(0.0)) / count("*")).as("qHead"),
      (sum(when(col("popularityGroup") === "M", 1.0).otherwise(0.0)) / count("*")).as("qMid"),
      (sum(when(col("popularityGroup") === "T", 1.0).otherwise(0.0)) / count("*")).as("qTail")
    )
  val joined = profileData.join(recommendationMixes, Seq("userId"), "inner")
    .join(segmentData, Seq("userId"), "inner")
  val divergence = Seq("head" -> "qHead", "mid" -> "qMid", "tail" -> "qTail")
    .map { case (pName, qName) =>
      val p = col(pName)
      val q = col(qName)
      val m = (p + q) / 2.0
      (klContributionColumn(p, m) + klContributionColumn(q, m)) / 2.0
    }.reduce(_ + _)
  val rawScores = joined.withColumn("mismatch", divergence)
  require(rawScores.filter(col("mismatch").isNull || isnan(col("mismatch")) ||
    col("mismatch") < -epsilon || col("mismatch") > 1.0 + epsilon)
    .limit(1).count() == 0L, "Calculated mismatch is outside its numerical bounds.")
  // Clamp only rounding-sized excursions after rejecting larger violations.
  val userScores = rawScores.withColumn("mismatch",
    greatest(lit(0.0), least(lit(1.0), col("mismatch"))))
  requireUserCoverage(userScores, cohort, "computed scores")
  validateUserMismatchScores(userScores)
  val segmentScores = userScores.groupBy("segment")
    .agg(count("*").as("userCount"), avg("mismatch").as("upd"))
  val overallScore = segmentScores.agg(avg("upd").as("overallUPD"))
  (userScores, segmentScores, overallScore)
}


For each selected test export, run `evaluateUPD` with `listSize = 10`. Match the evaluator overall score to the `all` row and each segment score to the `segment` row in `selected_on_validation.csv`.

In [ ]:
import spark.implicits._

val selectedResults = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv(new java.io.File(calibrationDir, "selected_on_validation.csv").getPath)
val settingDirs = calibrationDir.listFiles()
  .filter(file => file.isDirectory && file.getName.startsWith("top-10-"))
  .sortBy(_.getName)
val settingNames = settingDirs.map(_.getName.stripPrefix("top-10-")).toSeq
assert(settingNames.nonEmpty && settingNames.distinct.size == settingNames.size,
  s"Expected selected top-10 folders, found ${settingNames.size}: $settingNames")
val resultSettings = selectedResults.select("setting").distinct().as[String].collect().toSet
assert(resultSettings == settingNames.toSet,
  s"Settings in selected_on_validation.csv do not match selected top-10 folders: results=$resultSettings, folders=${settingNames.toSet}")
println(s"PASS: found ${settingNames.size} validation-selected test settings")

val tolerance = 1e-9
val segmentNames = Seq("blockbuster-focused", "diverse", "niche-focused")
var comparisonRows = Seq.empty[(String, Double, Double, Double, String)]

def selectedTestUPD(setting: String, groupType: String, group: String): Double = {
  val rows = selectedResults
    .filter(col("setting") === setting && col("groupType") === groupType && col("group") === group)
    .select(col("testUPD").cast("double").as("testUPD"))
    .collect()
  require(rows.length == 1,
    s"Expected one selected_on_validation.csv row for setting=$setting, groupType=$groupType, group=$group; found ${rows.length}")
  rows(0).getAs[Double]("testUPD")
}

settingDirs.foreach { settingDir =>
  val setting = settingDir.getName.stripPrefix("top-10-")
  val recommendations = spark.read.parquet(settingDir.getPath)
  val (_, segmentScores, overallScore) = evaluateUPD(
    profiles, movieGroups, recommendations, segments, evaluationUsers, listSize = 10)
  val evaluatedOverall = overallScore.first().getAs[Double]("overallUPD")
  val recordedOverall = selectedTestUPD(setting, "all", "all")
  val overallDifference = math.abs(recordedOverall - evaluatedOverall)
  comparisonRows :+= ((s"$setting/all", recordedOverall, evaluatedOverall, overallDifference,
    if (overallDifference <= tolerance) "PASS" else "FAIL"))

  val evaluatedSegments = segmentScores.select("segment", "upd").collect()
    .map(row => row.getAs[String]("segment") -> row.getAs[Double]("upd")).toMap
  assert(evaluatedSegments.keySet == segmentNames.toSet,
    s"Unexpected evaluated segments for $setting: ${evaluatedSegments.keySet}")
  segmentNames.foreach { segment =>
    val recorded = selectedTestUPD(setting, "segment", segment)
    val evaluated = evaluatedSegments(segment)
    val difference = math.abs(recorded - evaluated)
    comparisonRows :+= ((s"$setting/$segment", recorded, evaluated, difference,
      if (difference <= tolerance) "PASS" else "FAIL"))
  }
}

val comparisonTable = comparisonRows.toDF(
  "setting", "notebookUPD", "evaluatorUPD", "absDiff", "passFail")
assert(comparisonRows.size == settingNames.size * 4,
  s"Expected ${settingNames.size * 4} overall/segment comparisons, found ${comparisonRows.size}")
comparisonTable.orderBy("setting").show(100, truncate = false)

val allComparisonsPass = comparisonRows.forall(_._5 == "PASS")
println(s"${if (allComparisonsPass) "PASS" else "FAIL"}: all ${comparisonRows.size} selected-test UPD comparisons within tolerance $tolerance")

val crosscheckPath = new java.io.File(calibrationDir, "upd-crosscheck.csv").toPath
val writer = java.nio.file.Files.newBufferedWriter(
  crosscheckPath,
  java.nio.charset.StandardCharsets.UTF_8,
  java.nio.file.StandardOpenOption.CREATE_NEW,
  java.nio.file.StandardOpenOption.WRITE)
try {
  writer.write("setting,notebookUPD,evaluatorUPD,absDiff,passFail")
  writer.newLine()
  comparisonRows.foreach { case (setting, notebook, evaluator, difference, result) =>
    writer.write(Seq(setting, notebook.toString, evaluator.toString, difference.toString, result).mkString(","))
    writer.newLine()
  }
} finally {
  writer.close()
}
println(s"PASS: saved ${comparisonRows.size} rows to ${crosscheckPath.toString}")
assert(allComparisonsPass,
  s"One or more selected test UPD comparisons exceeded $tolerance; see ${crosscheckPath.toString}")


## Stop Spark

Release the local Spark session after all comparisons and the CSV export have succeeded.


In [ ]:
spark.stop()
println("PASS: Spark session stopped")
